# Liver dataset selection — audit (`_repaired`)

**v2 — przepisane na właściwe funkcje biblioteczne.** Tak jak `kidney_dataset_repaired.ipynb`: cała detekcja duplikatów/QC/morfologii, którą poprzednio pisałem ręcznie w komórkach, jest teraz w `msi_dataset_manager.exploration.dataset_review` (`DatasetReview`/`DatasetReviewProfile`/`DatasetExplorer.review_current()`/`.apply_review()`). Liver **ma już wbudowany profil** (`profile="liver"` w `_PROFILES`) — w przeciwieństwie do kidney, nic nie trzeba tu dosuwać ręcznie z poziomu notebooka.

**Ważna różnica względem kidney: liver nigdy nie przeszedł ręcznego przeglądu pod kątem pseudoreplikacji** — `filter.json` ma `exclude_dataset_ids: []`. Obecne 18 datasetów zostało zaakceptowanych wprost z zapytania.

Świadomie bez zmian względem poprzedniej wersji: brak analizy wspólnego zakresu m/z między narządami, brak zmian w bibliotece, brak nowych pobrań surowych danych.

In [1]:
import os
from pathlib import Path

current_path = Path.cwd().resolve()
repository_root = next(
    path
    for path in (current_path, *current_path.parents)
    if (path / "pyproject.toml").is_file()
)
os.chdir(repository_root)

repository_root

PosixPath('/home/max/repositories/MSIAutoEncoderWrapper')

In [2]:
import json

import pandas as pd
from IPython.display import display

from msi_dataset_manager.exploration import DatasetExplorer

# REMARK: date i download DB is 12.08.2026 (DD, MM, YYYY) -- same cache as the other two notebooks.
explorer = DatasetExplorer(
    source="metaspace",
    cache_dir="assets/local/datasets/metaspace",
    refresh_cache=False,
)

## 1. Szeroka pula kandydatów (ten sam filtr biologiczny co dotychczasowy `liver_dataset.1.ipynb`)

`condition="Wildtype"`, bez `mz_min`/`mz_max` — żeby audyt objął też rekordy odrzucane dziś przez filtr `200–1400`.

In [3]:
broad_filters = {
    "organism": "Mouse",
    "organism_part": "Liver",
    "condition": "Wildtype",
    "polarity": "Negative",
    "annotation_fdr": 0.1,
    "min_annotation_count": 1,
}
results = explorer.filter(broad_filters)
print(f"Found {len(results)} datasets")
display(results[["dataset_id", "name", "analyzer_type", "ionisation_source", "mz_min", "mz_max", "pixel_count"]])

METASPACE discovery:   0%|          | 0/3 [00:00<?, ?stage/s]

Current operation:   0%|          | 0/1 [00:00<?, ?operation/s]

Found 82 datasets


,dataset_id,name,analyzer_type,ionisation_source,mz_min,mz_max,pixel_count
0,2026-06-22_15h21m26s,WT_D12_3-neg,TOF,DESI,7.500978e+01,1199.833220,13964
1,2026-06-22_15h19m59s,WT_D12_2-neg,TOF,DESI,7.500978e+01,1199.833220,5142
2,2026-06-22_15h19m49s,WT_D12_1-neg,TOF,DESI,7.500978e+01,1199.833220,9002
3,2026-06-22_15h19m03s,WT_D5_3-neg,TOF,DESI,7.500978e+01,1199.833220,9488
4,2026-06-22_15h17m46s,WT_D5_2-neg,TOF,DESI,7.500978e+01,1199.833220,7172
...,...,...,...,...,...,...,...
77,2022-04-14_15h07m57s,2022-04-14_ME_DKFZACLY_S1_W4_DANneg_s10a33_100...,Orbitrap,AP-SMALDI5,9.900769e+01,404.029861,10000
78,2022-04-14_14h45m46s,2022-04-13_ME_DKFZACLY_S1_W8_DANneg_s10a33_100...,Orbitrap,AP-SMALDI5,9.900768e+01,404.029265,10000
79,2017-02-23_09h51m18s,Mouse liver_DMAN_200x200_25um_rcal,Orbitrap,MALDI,2.499993e+02,999.884929,40000
80,2017-02-22_15h01m27s,210217_mouseliver_DMAN_negative_200x200_25um-2...,Orbitrap,MALDI,2.500000e+02,999.881226,40000


## 2. Ładowanie dotychczasowej selekcji i przeniesienie jej wykluczeń do sesji

`data/liver_workspace/configs/datasets/liver/filter.json` ma `exclude_dataset_ids: []` (0 wykluczeń — brak wcześniejszej rundy przeglądu). Przenoszę to (pusty zbiór) do sesji dla spójności z kidney — jeśli kiedyś dopiszesz tam ręczne wykluczenia, ten notebook automatycznie je uwzględni.

In [4]:
existing_filter = json.load(open("data/liver_workspace/configs/datasets/liver/filter.json"))
existing_selection = json.load(open("data/liver_workspace/configs/datasets/liver/selection.json"))
existing_excluded_ids = existing_filter.get("exclude_dataset_ids", [])
existing_selected_ids = set(existing_selection["dataset_ids"])
print(f"existing selection: {len(existing_selected_ids)} selected, {len(existing_excluded_ids)} manually excluded")

explorer.exclude(existing_excluded_ids)

existing selection: 18 selected, 0 manually excluded


,dataset_id,name,source,project_accession,project_url,organisms,organism_parts,condition,growth_conditions,diseases,...,unannotated_pixel_count,annotated_pixel_fraction,annotation_fdr,spatial_annotation_count,spatial_annotation_database_count,spatial_stats_status,molecule_count,unique_molecule_count,unique_molecules,excluded
0,2026-06-22_15h21m26s,WT_D12_3-neg,metaspace,None,https://metaspace2020.eu/dataset/2026-06-22_15...,Mus musculus (mouse),Liver,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
1,2026-06-22_15h19m59s,WT_D12_2-neg,metaspace,None,https://metaspace2020.eu/dataset/2026-06-22_15...,Mus musculus (mouse),Liver,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
2,2026-06-22_15h19m49s,WT_D12_1-neg,metaspace,None,https://metaspace2020.eu/dataset/2026-06-22_15...,Mus musculus (mouse),Liver,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
3,2026-06-22_15h19m03s,WT_D5_3-neg,metaspace,None,https://metaspace2020.eu/dataset/2026-06-22_15...,Mus musculus (mouse),Liver,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
4,2026-06-22_15h17m46s,WT_D5_2-neg,metaspace,None,https://metaspace2020.eu/dataset/2026-06-22_15...,Mus musculus (mouse),Liver,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77,2022-04-14_15h07m57s,2022-04-14_ME_DKFZACLY_S1_W4_DANneg_s10a33_100...,metaspace,None,https://metaspace2020.eu/dataset/2022-04-14_15...,Mus musculus (mouse),Liver,Wildtype,Labeled,,...,None,None,0.1,None,None,None,None,None,,False
78,2022-04-14_14h45m46s,2022-04-13_ME_DKFZACLY_S1_W8_DANneg_s10a33_100...,metaspace,None,https://metaspace2020.eu/dataset/2022-04-14_14...,Mus musculus (mouse),Liver,Wildtype,Unlabeled,,...,None,None,0.1,None,None,None,None,None,,False
79,2017-02-23_09h51m18s,Mouse liver_DMAN_200x200_25um_rcal,metaspace,None,https://metaspace2020.eu/dataset/2017-02-23_09...,Mus musculus (mouse),Liver,Wildtype,N/A,,...,None,None,0.1,None,None,None,None,None,,False
80,2017-02-22_15h01m27s,210217_mouseliver_DMAN_negative_200x200_25um-2...,metaspace,None,https://metaspace2020.eu/dataset/2017-02-22_15...,Mus musculus (mouse),Liver,Wildtype,Caged,,...,None,None,0.1,None,None,None,None,None,,False


## 3. Przegląd biblioteczny (`DatasetExplorer.review_current`, `profile="liver"`)

Wbudowany profil liver (`low_pixel_threshold=200`, wzorzec morfologii `lobe`/`periportal`/`pericentral`/`zonation`/`capsule`/`portal`) — dokładnie te same wartości, które wcześniej ustaliłem ręcznie i skalibrowałem do skali liver (nie brain/kidney, patrz poprzednia wersja tego notebooka).

In [5]:
review = explorer.review_current(profile="liver")

print("available rules:", review.available_rules)
display(review.summary())

display(
    review.table.loc[
        review.table["duplicate_cluster_size"] > 1,
        ["duplicate_cluster_id", "dataset_id", "name", "pixel_count", "duplicate_confidence", "duplicate_excluded", "recommended_keeper_dataset_id"],
    ].sort_values(["duplicate_confidence", "duplicate_cluster_id"])
)
display(review.table.loc[review.table["low_pixel_flag"], ["dataset_id", "name", "pixel_count", "duplicate_excluded"]])

available rules: ('high_confidence_duplicates', 'mz_shift_qc_variants', 'explicit_regional_fragments')


,rule,dataset_count
0,high_confidence_duplicates,5
1,mz_shift_qc_variants,0
2,explicit_regional_fragments,0


,duplicate_cluster_id,dataset_id,name,pixel_count,duplicate_confidence,duplicate_excluded,recommended_keeper_dataset_id
56,technical-0029,2022-04-16_08h23m55s,2022-04-14_ME_DKFZACLY_S3_W8_DANneg_s10a33_100...,10000,ambiguous_shared_template,False,<NA>
57,technical-0029,2022-04-16_08h22m38s,2022-04-14_ME_DKFZACLY_S3_W4_DANneg_s10a33_100...,10000,ambiguous_shared_template,False,<NA>
75,technical-0029,2022-04-16_08h17m46s,2022-04-14_ME_DKFZACLY_S2_W8_DANneg_s10a33_100...,10000,ambiguous_shared_template,False,<NA>
76,technical-0029,2022-04-16_08h18m19s,2022-04-14_ME_DKFZACLY_S2_W4_DANneg_s10a33_100...,10000,ambiguous_shared_template,False,<NA>
78,technical-0029,2022-04-14_14h45m46s,2022-04-13_ME_DKFZACLY_S1_W8_DANneg_s10a33_100...,10000,ambiguous_shared_template,False,<NA>
22,technical-0048,2026-02-24_17h39m55s,2026_02_24_Rep-01,28745,ambiguous_shared_template,False,<NA>
23,technical-0048,2026-02-23_17h18m48s,2026_02_19_Rep-01,28745,ambiguous_shared_template,False,<NA>
10,technical-0056,2026-03-31_13h24m27s,id2_calnaf_quadraticen,34049,ambiguous_shared_template,False,<NA>
12,technical-0056,2026-03-30_16h51m11s,2026_03_25_calnaf_quadraticen,34049,ambiguous_shared_template,False,<NA>
44,technical-0001,2024-06-28_07h10m21s,liver rn lip tic-50ppm,110,high_confidence_duplicate,True,2024-06-26_03h54m35s


,dataset_id,name,pixel_count,duplicate_excluded
44,2024-06-28_07h10m21s,liver rn lip tic-50ppm,110,True
46,2024-06-26_03h54m35s,liver rn lip tic,110,False
48,2024-06-25_11h23m24s,liver storage day 5 tic,192,False


## 4. Zastosowanie reguł i finalna, poprawiona lista

Stosuję `high_confidence_duplicates` i `mz_shift_qc_variants` (0 dla liver) oraz `explicit_regional_fragments` dla spójności (profil liver ma pustą `explicit_regional_names`, więc to 0). Doradcze `morphology_hint` i `low_pixel_flag` zostają tylko w tabeli.

In [6]:
applied_rules = ["high_confidence_duplicates", "mz_shift_qc_variants", "explicit_regional_fragments"]
explorer.apply_review(review, rules=applied_rules)

final_filters = {
    "organism": "Mouse",
    "organism_part": "Liver",
    "polarity": "Negative",
    "condition": "Wildtype",
    "mz_min": 200,
    "mz_max": 1400,
    "annotation_fdr": 0.1,
    "min_annotation_count": 1,
    "include_molecule_stats": True,
    "include_spatial_annotation_stats": False,  # see brain_dataset.ipynb section 6 for the cost rationale
}
# exclude_dataset_ids intentionally omitted -- session-level exclusions from step 2/4 persist across this re-query.
results_liver_repaired = explorer.filter(final_filters)
print(f"repaired liver shortlist: {len(results_liver_repaired)} datasets (previously {len(existing_selected_ids)})")
display(results_liver_repaired[["dataset_id", "name", "analyzer_type", "pixel_count", "molecule_count", "unique_molecule_count"]])

METASPACE discovery:   0%|          | 0/3 [00:00<?, ?stage/s]

Current operation:   0%|          | 0/1 [00:00<?, ?operation/s]

repaired liver shortlist: 18 datasets (previously 18)


,dataset_id,name,analyzer_type,pixel_count,molecule_count,unique_molecule_count
0,2025-08-19_19h29m48s,Lipids3_top_control_nodiverter,Orbitrap,511,207,5
1,2025-08-19_19h33m37s,Lipids4_top_Nh4F_diverter,Orbitrap,670,37,0
2,2025-08-19_19h33m18s,Lipids4_bottom_control_nodiverter,Orbitrap,681,109,6
3,2025-08-19_19h29m30s,Lipids3_bottom_Nh4F_diverter,Orbitrap,508,182,10
4,2025-08-19_19h28m58s,Lipids2_top_NH4F_diverter,Orbitrap,534,225,12
5,2025-08-19_19h23m41s,Lipids1_bottom_NH4F_diverter,Orbitrap,754,262,44
6,2025-08-19_19h27m06s,Lipids2_bottom_control_nodiverter,Orbitrap,532,120,1
7,2025-08-19_19h26m22s,Lipids1_top_control_nodiverter,Orbitrap,653,216,9
8,2025-06-30_15h03m07s,Tissue5_top_NH4F,Orbitrap,854,829,149
9,2025-06-30_15h02m48s,Tissue4_top_NH4F,Orbitrap,839,468,23


## 5. Które datasety zostały wycięte względem dotychczasowego `liver/` — pełne porównanie

In [7]:
repaired_selected_ids = set(results_liver_repaired["dataset_id"].astype(str))
review_reasons = {
    dataset_id: "high_confidence_duplicate"
    for dataset_id in review.exclusion_ids(["high_confidence_duplicates"])
}
review_reasons.update({
    dataset_id: "mz_shift_qc_variant"
    for dataset_id in review.exclusion_ids(["mz_shift_qc_variants"])
})

comparison = results[["dataset_id", "name"]].copy()
comparison["in_original_selection"] = comparison["dataset_id"].isin(existing_selected_ids)
comparison["in_repaired_selection"] = comparison["dataset_id"].isin(repaired_selected_ids)
comparison["was_manually_excluded_before"] = comparison["dataset_id"].isin(existing_excluded_ids)
comparison["newly_excluded_reason"] = comparison["dataset_id"].map(review_reasons).fillna("")

changed = comparison.loc[comparison["in_original_selection"] != comparison["in_repaired_selection"]]
print(f"datasets whose accept/exclude status changed: {len(changed)}")
display(changed)

print(f"\ntotal: {comparison['in_original_selection'].sum()} (original) -> {comparison['in_repaired_selection'].sum()} (repaired)")

print("\nfor reference -- objectively flagged duplicates NOT in the original 18 either")
print("(only relevant if the corpus is later widened beyond the current 18):")
display(comparison.loc[(comparison["newly_excluded_reason"] != "") & ~comparison["in_original_selection"]])

datasets whose accept/exclude status changed: 0


,dataset_id,name,in_original_selection,in_repaired_selection,was_manually_excluded_before,newly_excluded_reason



total: 18 (original) -> 18 (repaired)

for reference -- objectively flagged duplicates NOT in the original 18 either
(only relevant if the corpus is later widened beyond the current 18):


,dataset_id,name,in_original_selection,in_repaired_selection,was_manually_excluded_before,newly_excluded_reason
43,2024-07-01_09h59m02s,liver storage day 2 tic,False,False,False,high_confidence_duplicate
44,2024-06-28_07h10m21s,liver rn lip tic-50ppm,False,False,False,high_confidence_duplicate
45,2024-06-26_03h47m05s,liver storage day 2 tic-110ppm,False,False,False,high_confidence_duplicate
47,2024-06-25_11h38m02s,liver storage day 2 50ppm,False,False,False,high_confidence_duplicate
54,2023-06-02_11h54m37s,liver 2-3 50um ms range 300_1500 213_306 pNA-,False,False,False,high_confidence_duplicate


In [8]:
output_path = Path("data/liver_workspace/configs/datasets/liver_repaired")
exported = explorer.export_selection(output_path, sort_by="download_size_bytes", ascending=False)
exported

{'filters': PosixPath('data/liver_workspace/configs/datasets/liver_repaired/filter.json'),
 'selection': PosixPath('data/liver_workspace/configs/datasets/liver_repaired/selection.json')}

## Podsumowanie

- Przepisane na `DatasetExplorer.review_current(profile="liver")`/`.apply_review()` — liver ma już wbudowany profil w bibliotece, więc zero kodu specyficznego dla narządu w tym notebooku (w przeciwieństwie do kidney).
- Zmiana względem dotychczasowej selekcji: **0** — żaden z obiektywnie znalezionych 5 duplikatów nie był częścią pobranego korpusu 18, więc poprawiona lista wychodzi identyczna liczebnie. Audyt ma tu wartość prewencyjną/procesową (patrz sekcja 5, druga tabela) — istotną, jeśli kiedyś poszerzysz korpus liver ponad obecne 18.
- Pełna tabela zmian w sekcji 5.
- Eksport do `data/liver_workspace/configs/datasets/liver_repaired/` — istniejący `liver/` nie został nadpisany.